# Fine-tune the Kurrent adapter on Kade lines, right here in Jupyter

This notebook trains on **this notebook instance's own GPU**: no training job, the output appears
below the cells. It runs the same script as `sagemaker_kade_lora.ipynb`
(`sagemaker/code/finetune_qwen_kurrent_lora.py`), so results from the two are comparable.
`docs/lora-finetuning-explained.md` explains every step in plain language.

**The notebook instance must have a GPU.** When you create it (SageMaker → Notebook instances →
Create):

| Setting | Value |
|---|---|
| Notebook instance type | `ml.g4dn.xlarge` (T4, 16 GB, about $0.74/h) or `ml.g5.xlarge` (A10G, 24 GB, about $1.41/h) |
| Volume size | **50 GB** (the default 5 GB can't hold the 9 GB model) |
| Kernel, once it's running | `conda_pytorch` (any `conda_pytorch_*`) |

**A GPU notebook instance bills every hour it is running, even with the browser closed.** Stop it
(SageMaker → Notebook instances → Stop) as soon as you're done. Forgotten over a weekend, a g5 is
about $70.

Run the cells from top to bottom. Start with `RUN = "smoke"` in step 4.

## 1. Install the libraries

Qwen3-VL needs `transformers` 4.57 or newer. Packages you install are wiped every time the instance is stopped, so run this cell again after each start. Run it **before** anything imports `transformers`; if you already did, restart the kernel (Kernel → Restart) and start here.

In [ ]:
%pip install -q "transformers==4.57.6" "peft==0.21.2" "accelerate>=1.10"

## 2. Check the GPU and the disk

In [ ]:
import os
import shutil
import torch

WORK = "/home/ec2-user/SageMaker/kade-lora"   # on the instance's volume: survives a Stop, not a Delete
os.makedirs(WORK, exist_ok=True)
# Keep the 9 GB model download on the big volume, not the small system disk.
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Stop this instance and change its type to ml.g4dn.xlarge or ml.g5.xlarge.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU:  {gpu.name}, {gpu.total_memory / 1e9:.0f} GB, "
      f"{'bf16' if torch.cuda.get_device_capability()[0] >= 8 else 'fp16 (no bf16 on this GPU)'}")
free = shutil.disk_usage(WORK).free / 1e9
print(f"Disk: {free:.0f} GB free in {WORK}")
if free < 20:
    print("WARNING: under 20 GB free. Stop the instance and raise its volume size to 50 GB.")

## 3. Get the data

You need `kade_lines.zip`: the line crops made on Ubuntu with `notebooks/kade_line_segmentation.ipynb`
(`docs/ubuntu-setup.md`), zipped with
`cd ~/badger-scribe-data && python -m zipfile -c kade_lines.zip kade_lines/`. Drag it into this
folder in the Jupyter file browser on the left.

`train.csv` comes from the class's shared bucket. If you can't reach it, drag `train.csv` in too.

This pairs each crop with its line of the label, for pages where kraken found as many lines as the
label has (the `Postmark` line doesn't count; it isn't in the image). The dev pages, all of
`kade_112`, are never trained on. **Check the first printed line:** if well under 70% of training
pages are usable, look at the segmentation before training.

In [ ]:
import sys

repo_dir = os.path.abspath("..")   # this notebook lives in <repo>/notebooks
script = os.path.join(repo_dir, "sagemaker", "code", "finetune_qwen_kurrent_lora.py")
sys.path.insert(0, os.path.join(repo_dir, "sagemaker"))
import prepare_kade_lines

lines_zip = "kade_lines.zip"       # where you dragged it
train_csv = os.path.join(WORK, "train.csv")
if os.path.exists("train.csv"):
    shutil.copy("train.csv", train_csv)
elif not os.path.exists(train_csv):
    import boto3
    boto3.client("s3").download_file("ml-marathon-shared", "badger-scribe/train.csv", train_csv)

data_dir = os.path.join(WORK, "data")
stats = prepare_kade_lines.build(lines_zip, train_csv, data_dir)

## 4. Choose the run and the settings

| Run | What it does | About how long |
|---|---|---|
| `smoke` | 64 lines, 1 epoch, 2 dev pages: checks everything works | 10–15 min (mostly the model download, once) |
| `baseline` | No training; scores the 11 dev pages with the Kurrent adapter as it is | 5–10 min |
| `full` | All lines, scored after every epoch | Roughly 30–40 min on g5, 1–1.5 h on g4dn (an estimate until you've timed one) |

The settings are explained in `docs/lora-finetuning-explained.md`, "The settings you can change".
**Change one at a time.** Each combination gets its own folder; running the same combination
again **continues** it from its last saved epoch (useful after a crash). To start it over, delete
its folder under `kade-lora/runs/`.

In [ ]:
RUN = "smoke"   # "smoke", then "baseline", then "full"

settings = {
    "lr": 1e-4,           # learning rate: size of each nudge
    "epochs": 3,          # passes over all training lines
    "lora_r": 16,         # adapter size
    "lora_alpha": 32,     # adapter strength; keep at 2 x lora_r
    "lora_dropout": 0.05,
    "batch_size": 4,      # lines per step; halve it (and double grad_accum) if the GPU runs out of memory
    "grad_accum": 4,      # effective batch = batch_size x grad_accum = 16
    "min_pixels": 65536,  # smallest image size the model sees; higher = more detail, slower
}
if RUN == "smoke":
    settings.update(epochs=1, max_train_lines=64, dev_pages=2)
elif RUN == "baseline":
    settings = {"epochs": 0}

run_name = RUN if RUN != "full" else "full-" + "-".join(f"{k}{v}" for k, v in settings.items())
run_dir = os.path.join(WORK, "runs", run_name)
for sub in ("model", "checkpoints", "output"):
    os.makedirs(os.path.join(run_dir, sub), exist_ok=True)
print(run_dir)

## 5. Train

The log streams below and is also saved to `train.log` in the run folder. If the browser
disconnects, the training keeps going as long as the instance is running; reopen the notebook and
run `!tail -n 30 {log}` to see where it got to. To stop it, use Kernel → Interrupt.

Lines to look for: `[dev epoch0] ... mean page CER` (the score before training), `loss` going down,
and a `[dev epochN]` line after each epoch.

In [ ]:
args = {**settings,
        "train_dir": data_dir,
        "model_dir": os.path.join(run_dir, "model"),
        "checkpoint_dir": os.path.join(run_dir, "checkpoints"),
        "output_data_dir": os.path.join(run_dir, "output")}
flags = " ".join(f"--{k} {v}" for k, v in args.items())
log = os.path.join(run_dir, "train.log")
python = sys.executable   # this kernel's own Python, where step 1 installed the libraries
!{python} {script} {flags} 2>&1 | tee -a {log}

## 6. Results

Epoch 0 is the Kurrent adapter before training. The job kept the adapter from the best epoch in `model/`.

In [ ]:
import json
import pandas as pd

out = os.path.join(run_dir, "model")
summary = json.load(open(os.path.join(out, "train_summary.json")))
print(json.dumps(summary, indent=2))
print(f"\nKade dev CER: {summary['dev_cer_before']:.3f} before -> {summary['dev_cer_best']:.3f} best "
      f"(epoch {summary['best_epoch']}). Old base-model result: about 0.5.")

log_json = json.load(open(os.path.join(out, "train_log.json")))
curve = pd.DataFrame(log_json["dev_cer_by_epoch"])
print(curve.to_string(index=False))
if log_json["loss"]:
    import matplotlib.pyplot as plt
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
    loss = pd.DataFrame(log_json["loss"])
    a.plot(loss.step, loss.loss); a.set_xlabel("step"); a.set_ylabel("training loss")
    b.plot(curve.epoch, curve.dev_cer, marker="o"); b.set_xlabel("epoch"); b.set_ylabel("dev CER (lower is better)")
    plt.tight_layout(); plt.show()

In [ ]:
before = pd.read_csv(os.path.join(out, "eval_epoch0.csv"), keep_default_na=False)
best = pd.read_csv(os.path.join(out, f"eval_epoch{summary['best_epoch']}.csv"), keep_default_na=False)
cmp = before[["page_id", "cer"]].merge(best[["page_id", "cer"]], on="page_id", suffixes=("_before", "_best"))
cmp["gain"] = cmp.cer_before - cmp.cer_best
print(cmp.round(3).to_string(index=False))

page = cmp.sort_values("gain").iloc[-1].page_id   # the page that improved most
print(f"\n{page}\nREFERENCE\n" + before.set_index("page_id").loc[page, "text_ref"][:600])
print("\nBEFORE\n" + before.set_index("page_id").loc[page, "text_pred"][:600])
print("\nBEST\n" + best.set_index("page_id").loc[page, "text_pred"][:600])

## 7. Save the adapter to S3 (optional), then stop the instance

The run folder survives a Stop, but not a Delete, and teammates can't see it. Copy it to your
bucket to keep it. The adapter is about 130 MB; checkpoints are left out.

In [ ]:
bucket_name = "sinkorswim-doejohn"   # <-- EDIT ME: your own bucket
!aws s3 sync {run_dir} s3://{bucket_name}/badger-scribe/kade-lora/runs/{run_name}/ --exclude "checkpoints/*"

**Now stop the instance:** SageMaker → Notebook instances → select it → Actions → Stop.
Everything under `/home/ec2-user/SageMaker` (this repo, `kade-lora/`) is still there when you
start it again; re-run step 1 after each start.